# Notebook 03: Error Handling & Validation Test
## Demonstrates the quarantine pattern and fail-fast behaviour

This notebook was included to test the pipeline's error handling by intentionally injecting 
bad rows and confirming they are caught, quarantined, and excluded from 
downstream processing. It also demonstrates the fail-fast check that halts 
the pipeline if too many rows are lost.

In [0]:
# Configuration
STORAGE_ACCOUNT = "stcw2bdcbh"
STORAGE_KEY = dbutils.secrets.get(scope="kv-scope", key="storage-account-key")

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)

RAW_PATH = f"abfss://raw@{STORAGE_ACCOUNT}.dfs.core.windows.net/amazon-reviews/Reviews.csv"
QUARANTINE_PATH = f"abfss://quarantine@{STORAGE_ACCOUNT}.dfs.core.windows.net/rule=schema_violation/"

print("Configuration set.")

Configuration set.


### Step 1: Create Test Data
Four rows are created manually with two valid, one with a null Score, 
and one with a null Id to represent examples of malformed records 
that could arrive in a real ingestion scenario.

In [0]:
# Create a small test dataframe with deliberately bad rows
# Row 1: valid row
# Row 2: null Score (should be quarantined)
# Row 3: null Id (should be quarantined)
# Row 4: valid row

from pyspark.sql.types import StructType, StructField, StringType, IntegerType

test_schema = StructType([
    StructField("Id", IntegerType(), True),
    StructField("ProductId", StringType(), True),
    StructField("UserId", StringType(), True),
    StructField("ProfileName", StringType(), True),
    StructField("HelpfulnessNumerator", IntegerType(), True),
    StructField("HelpfulnessDenominator", IntegerType(), True),
    StructField("Score", IntegerType(), True),
    StructField("Time", IntegerType(), True),
    StructField("Summary", StringType(), True),
    StructField("Text", StringType(), True)
])

test_data = [
    (1, "B001", "U001", "Alice", 1, 1, 5, 1234567890, "Great", "Loved it"),
    (2, "B002", "U002", "Bob", 0, 0, None, 1234567890, "Bad", "Hated it"),
    (None, "B003", "U003", "Charlie", 2, 3, 4, 1234567890, "OK", "It was fine"),
    (4, "B004", "U004", "Dave", 3, 3, 3, 1234567890, "Neutral", "Meh")
]

df_test = spark.createDataFrame(test_data, test_schema)
print(f"Test dataframe created with {df_test.count()} rows")
df_test.show()

Test dataframe created with 4 rows
+----+---------+------+-----------+--------------------+----------------------+-----+----------+-------+-----------+
|  Id|ProductId|UserId|ProfileName|HelpfulnessNumerator|HelpfulnessDenominator|Score|      Time|Summary|       Text|
+----+---------+------+-----------+--------------------+----------------------+-----+----------+-------+-----------+
|   1|     B001|  U001|      Alice|                   1|                     1|    5|1234567890|  Great|   Loved it|
|   2|     B002|  U002|        Bob|                   0|                     0| NULL|1234567890|    Bad|   Hated it|
|NULL|     B003|  U003|    Charlie|                   2|                     3|    4|1234567890|     OK|It was fine|
|   4|     B004|  U004|       Dave|                   3|                     3|    3|1234567890|Neutral|        Meh|
+----+---------+------+-----------+--------------------+----------------------+-----+----------+-------+-----------+



### Step 2: Quarantine Invalid Rows
Rows missing a Score or Id are filtered out and written to the quarantine 
zone under `rule=schema_violation/`. Valid rows pass through to the next stage.

In [0]:
# Run quarantine logic on test data
# Rows with null Score or null Id are quarantined
from pyspark.sql.functions import col

df_quarantine = df_test.filter(col("Score").isNull() | col("Id").isNull())
quarantine_count = df_quarantine.count()

print(f"Rows quarantined: {quarantine_count}")
df_quarantine.show()

# Write bad rows to quarantine zone
df_quarantine.write.mode("overwrite").parquet(QUARANTINE_PATH)
print(f"Quarantine written to: {QUARANTINE_PATH}")

# Valid rows only
df_valid = df_test.filter(col("Score").isNotNull() & col("Id").isNotNull())
valid_count = df_valid.count()
print(f"Valid rows remaining: {valid_count}")
df_valid.show()

Rows quarantined: 2
+----+---------+------+-----------+--------------------+----------------------+-----+----------+-------+-----------+
|  Id|ProductId|UserId|ProfileName|HelpfulnessNumerator|HelpfulnessDenominator|Score|      Time|Summary|       Text|
+----+---------+------+-----------+--------------------+----------------------+-----+----------+-------+-----------+
|   2|     B002|  U002|        Bob|                   0|                     0| NULL|1234567890|    Bad|   Hated it|
|NULL|     B003|  U003|    Charlie|                   2|                     3|    4|1234567890|     OK|It was fine|
+----+---------+------+-----------+--------------------+----------------------+-----+----------+-------+-----------+

Quarantine written to: abfss://quarantine@stcw2bdcbh.dfs.core.windows.net/rule=schema_violation/
Valid rows remaining: 2
+---+---------+------+-----------+--------------------+----------------------+-----+----------+-------+--------+
| Id|ProductId|UserId|ProfileName|Helpfulne

### Step 3: Fail-Fast Check (Boundary Case)
Tests the 50% row loss threshold. In this test exactly 50% of rows were 
quarantined and so this sits on the boundary and therefore correctly passes through.

In [0]:
# Demonstrate fail-fast behaviour
# If more than 50% of rows are lost, the pipeline raises an exception

raw_count = df_test.count()
valid_count = df_valid.count()

print(f"Raw rows: {raw_count}")
print(f"Valid rows: {valid_count}")
print(f"Row loss: {((raw_count - valid_count) / raw_count) * 100:.1f}%")

# In our test, 50% of rows were lost - this should trigger the fail-fast
if valid_count < raw_count * 0.5:
    raise Exception(
        f"FAIL: Too many rows lost. Raw: {raw_count}, Valid: {valid_count}. "
        f"Pipeline halted to prevent corrupt output."
    )

print("Row count check passed.")

Raw rows: 4
Valid rows: 2
Row loss: 50.0%
Row count check passed.


### Step 4: Fail-Fast Check (Triggered)
Simulates a scenario where 60% of rows are lost. The pipeline raises an 
exception with a clear message and halts. This is done to prevent corrupt or incomplete 
data from reaching the processed zone.

In [0]:
# Demonstrate fail-fast triggering
# Simulate a scenario where more than 50% of rows are lost

raw_count_demo = 10
valid_count_demo = 4  # 60% loss, should trigger fail-fast

print(f"Simulating: {raw_count_demo} raw rows, {valid_count_demo} valid rows")
print(f"Row loss: {((raw_count_demo - valid_count_demo) / raw_count_demo) * 100:.1f}%")

if valid_count_demo < raw_count_demo * 0.5:
    raise Exception(
        f"FAIL: Too many rows lost. Raw: {raw_count_demo}, Valid: {valid_count_demo}. "
        f"Pipeline halted to prevent corrupt output."
    )

Simulating: 10 raw rows, 4 valid rows
Row loss: 60.0%


---------------------------------------------------------------------------
Exception                                 Traceback (most recent call last)
File <command-4600042924246192>, line 11
      8 print(f"Row loss: {((raw_count_demo - valid_count_demo) / raw_count_demo) * 100:.1f}%")
     10 if valid_count_demo < raw_count_demo * 0.5:
---> 11     raise Exception(
     12         f"FAIL: Too many rows lost. Raw: {raw_count_demo}, Valid: {valid_count_demo}. "
     13         f"Pipeline halted to prevent corrupt output."
     14     )

Exception: FAIL: Too many rows lost. Raw: 10, Valid: 4. Pipeline halted to prevent corrupt output.